In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.utils import resample

In [ ]:
users_code_location_df = pd.read_csv('users_code_location.csv', index_col=0)

users_code_location_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 63572 entries, 0 to 63571
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Unnamed: 0     63572 non-null  int64 
 1   Username       63572 non-null  object
 2   Display_Name   63572 non-null  object
 3   Gender         63572 non-null  object
 4   notebook_url   63572 non-null  object
 5   code_location  63572 non-null  object
 6   labels         58757 non-null  object
dtypes: int64(1), object(6)
memory usage: 3.9+ MB


In [ ]:
users_code_location_df.drop('Unnamed: 0', axis=1, inplace=True)
users_code_location_df

,Username,Display_Name,Gender,notebook_url,code_location,labels
0,aamaia,amaia,female,https://www.kaggle.com/code/aamaia/display-sample,D:\learning\thesis\female\aamaia\display-sampl...,['Carvana Image Masking Challenge']
1,aamaia,amaia,female,https://www.kaggle.com/code/aamaia/emojis-glove,D:\learning\thesis\female\aamaia\emojis-glove....,['GloVe: Global Vectors for Word Representatio...
2,aamaia,amaia,female,https://www.kaggle.com/code/aamaia/failures-in...,D:\learning\thesis\female\aamaia\failures-in-p...,['Bosch Production Line Performance']
3,aamaia,amaia,female,https://www.kaggle.com/code/aamaia/large-vehicles,D:\learning\thesis\female\aamaia\large-vehicle...,['Dstl Satellite Imagery Feature Detection']
4,aamaia,amaia,female,https://www.kaggle.com/code/aamaia/leeak,D:\learning\thesis\female\aamaia\leeak.ipynb,['Intel &amp; MobileODT Cervical Cancer Screen...
...,...,...,...,...,...,...
63567,zzettrkalpakbal,Izzet Turkalp Akbasli,male,https://www.kaggle.com/code/zzettrkalpakbal/ti...,D:\learning\thesis\male\zzettrkalpakbal\titani...,['Titanic - Machine Learning from Disaster']
63568,zzettrkalpakbal,Izzet Turkalp Akbasli,male,https://www.kaggle.com/code/zzettrkalpakbal/tr...,D:\learning\thesis\male\zzettrkalpakbal\transf...,"['Medical MNIST', '[Private Datasource]']"
63569,zzettrkalpakbal,Izzet Turkalp Akbasli,male,https://www.kaggle.com/code/zzettrkalpakbal/tu...,D:\learning\thesis\male\zzettrkalpakbal\tutori...,['Brain stroke prediction dataset']
63570,zzettrkalpakbal,Izzet Turkalp Akbasli,male,https://www.kaggle.com/code/zzettrkalpakbal/un...,D:\learning\thesis\male\zzettrkalpakbal\unifor...,['BU YARIŞMA SAYFASI İPTAL EDİLMİŞTİR']


In [ ]:
users_code_location_df = users_code_location_df.dropna()
users_code_location_df = users_code_location_df[~users_code_location_df.notebook_url.str.split('/').str[-1].str.startswith('notebook')]
users_code_location_df['labels'] = users_code_location_df['labels'].map(lambda x: ast.literal_eval(x))
users_code_location_df['labels'] = users_code_location_df['labels'].apply(lambda x: [value for value in x if value not in ['[Private Datasource]','No attached data sources']] if x is not None else x)
users_code_location_df['labels'] = users_code_location_df['labels'].apply(lambda x: x if len(x) > 0 else None)
users_code_location_df = users_code_location_df.dropna()

exploded_df = users_code_location_df.explode('labels')

In [ ]:
users_code_location_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 50588 entries, 0 to 63570
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Username       50588 non-null  object
 1   Display_Name   50588 non-null  object
 2   Gender         50588 non-null  object
 3   notebook_url   50588 non-null  object
 4   code_location  50588 non-null  object
 5   labels         50588 non-null  object
dtypes: object(6)
memory usage: 2.7+ MB


In [ ]:
exploded_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 63711 entries, 0 to 63570
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Username       63711 non-null  object
 1   Display_Name   63711 non-null  object
 2   Gender         63711 non-null  object
 3   notebook_url   63711 non-null  object
 4   code_location  63711 non-null  object
 5   labels         63711 non-null  object
dtypes: object(6)
memory usage: 3.4+ MB


In [ ]:
grouped_df = exploded_df.groupby(['Gender', 'labels']).size().reset_index(name='count')
grouped_df = grouped_df[~grouped_df.labels.isin(['[Private Datasource]','No attached data sources'])]

In [ ]:
grouped_df.head()

,Gender,labels,count
0,female,Logistic regression To predict heart disease,2
1,female,Arabic Article Headline Generation,1
2,female,Basic Cars Characteristics 🏎,1
3,female,Diagnosis of COVID-19 and its clinical spectrum,1
4,female,E-Commerce Shipping Data,6


In [ ]:
grouped_df.nlargest(25,'count')

,Gender,labels,count
14223,male,Titanic - Machine Learning from Disaster,1511
6836,male,Digit Recognizer,743
8947,male,House Prices - Advanced Regression Techniques,729
11178,male,Natural Language Processing with Disaster Tweets,355
8974,male,Housing Prices Competition for Kaggle Learn Users,285
6279,male,Credit Card Fraud Detection,262
13340,male,Spaceship Titanic,256
9550,male,Iris Species,255
2756,female,Titanic - Machine Learning from Disaster,242
6005,male,CommonLit Readability Prize,204


In [ ]:
users_code_location_df['top_labels'] = users_code_location_df.labels.apply(
    lambda labels: set(grouped_df.nlargest(25,'count').labels.values).intersection(labels))

In [ ]:
users_code_location_df

,Username,Display_Name,Gender,notebook_url,code_location,labels,top_labels
0,aamaia,amaia,female,https://www.kaggle.com/code/aamaia/display-sample,D:\learning\thesis\female\aamaia\display-sampl...,[Carvana Image Masking Challenge],{}
1,aamaia,amaia,female,https://www.kaggle.com/code/aamaia/emojis-glove,D:\learning\thesis\female\aamaia\emojis-glove....,[GloVe: Global Vectors for Word Representation...,{}
2,aamaia,amaia,female,https://www.kaggle.com/code/aamaia/failures-in...,D:\learning\thesis\female\aamaia\failures-in-p...,[Bosch Production Line Performance],{}
3,aamaia,amaia,female,https://www.kaggle.com/code/aamaia/large-vehicles,D:\learning\thesis\female\aamaia\large-vehicle...,[Dstl Satellite Imagery Feature Detection],{}
4,aamaia,amaia,female,https://www.kaggle.com/code/aamaia/leeak,D:\learning\thesis\female\aamaia\leeak.ipynb,[Intel &amp; MobileODT Cervical Cancer Screening],{}
...,...,...,...,...,...,...,...
63566,zzettrkalpakbal,Izzet Turkalp Akbasli,male,https://www.kaggle.com/code/zzettrkalpakbal/t-...,D:\learning\thesis\male\zzettrkalpakbal\t-rk-e...,[Türkçe isimler],{}
63567,zzettrkalpakbal,Izzet Turkalp Akbasli,male,https://www.kaggle.com/code/zzettrkalpakbal/ti...,D:\learning\thesis\male\zzettrkalpakbal\titani...,[Titanic - Machine Learning from Disaster],{Titanic - Machine Learning from Disaster}
63568,zzettrkalpakbal,Izzet Turkalp Akbasli,male,https://www.kaggle.com/code/zzettrkalpakbal/tr...,D:\learning\thesis\male\zzettrkalpakbal\transf...,[Medical MNIST],{}
63569,zzettrkalpakbal,Izzet Turkalp Akbasli,male,https://www.kaggle.com/code/zzettrkalpakbal/tu...,D:\learning\thesis\male\zzettrkalpakbal\tutori...,[Brain stroke prediction dataset],{}


In [ ]:
only_top_labels_df = users_code_location_df[users_code_location_df.top_labels.apply(lambda x: len(x) == 1)]

In [ ]:
only_top_labels_df

,Username,Display_Name,Gender,notebook_url,code_location,labels,top_labels
29,aashnaashahh1504,Aashna Shah,female,https://www.kaggle.com/code/aashnaashahh1504/g...,D:\learning\thesis\female\aashnaashahh1504\get...,[Titanic - Machine Learning from Disaster],{Titanic - Machine Learning from Disaster}
69,abireltaief,Abir ELTAIEF,female,https://www.kaggle.com/code/abireltaief/look-f...,D:\learning\thesis\female\abireltaief\look-for...,"[Notebooks of the Week - Hidden Gems, Meta Kag...",{Meta Kaggle}
86,abirpattnaik,Abir Pattnaik,female,https://www.kaggle.com/code/abirpattnaik/house...,D:\learning\thesis\female\abirpattnaik\house-p...,[House Prices - Advanced Regression Techniques],{House Prices - Advanced Regression Techniques}
88,abirpattnaik,Abir Pattnaik,female,https://www.kaggle.com/code/abirpattnaik/nlp-d...,D:\learning\thesis\female\abirpattnaik\nlp-dis...,[Natural Language Processing with Disaster Twe...,{Natural Language Processing with Disaster Twe...
108,adakibet,Ada Kibet,female,https://www.kaggle.com/code/adakibet/exercise-...,D:\learning\thesis\female\adakibet\exercise-pi...,[Housing Prices Competition for Kaggle Learn U...,{Housing Prices Competition for Kaggle Learn U...
...,...,...,...,...,...,...,...
63539,zyper26,Shubham Chaurasia,male,https://www.kaggle.com/code/zyper26/mnist-svm-...,D:\learning\thesis\male\zyper26\mnist-svm-xgbo...,[Digit Recognizer],{Digit Recognizer}
63542,zyper26,Shubham Chaurasia,male,https://www.kaggle.com/code/zyper26/titanic-ag...,D:\learning\thesis\male\zyper26\titanic-aggreg...,[Titanic - Machine Learning from Disaster],{Titanic - Machine Learning from Disaster}
63561,zzettrkalpakbal,Izzet Turkalp Akbasli,male,https://www.kaggle.com/code/zzettrkalpakbal/my...,D:\learning\thesis\male\zzettrkalpakbal\my-fir...,[Iris Species],{Iris Species}
63563,zzettrkalpakbal,Izzet Turkalp Akbasli,male,https://www.kaggle.com/code/zzettrkalpakbal/py...,D:\learning\thesis\male\zzettrkalpakbal\pycare...,"[Stroke Prediction Dataset, Brain stroke predi...",{Stroke Prediction Dataset}


In [ ]:
only_top_labels_df.groupby(['Gender']).size()

Gender
female     929
male      6724
dtype: int64

In [ ]:
NUMBER_OFF_SAMPLES_BOTH_GENDERS = 2000

In [ ]:
# Separate the data based on gender
female_data = only_top_labels_df[only_top_labels_df['Gender'] == 'female']
male_data = only_top_labels_df[only_top_labels_df['Gender'] == 'male']

In [ ]:
male_data.top_labels.value_counts() / len(male_data) * 100

{Titanic - Machine Learning from Disaster}              21.936347
{Digit Recognizer}                                      10.633551
{House Prices - Advanced Regression Techniques}         10.499703
{Natural Language Processing with Disaster Tweets}       5.279595
{Spaceship Titanic}                                      3.777513
{Credit Card Fraud Detection}                            3.762641
{Housing Prices Competition for Kaggle Learn Users}      3.673409
{Iris Species}                                           3.509816
{CommonLit Readability Prize}                            3.033908
{Mechanisms of Action (MoA) Prediction}                  3.019036
{Pima Indians Diabetes Database}                         2.781083
{Breast Cancer Wisconsin (Diagnostic) Data Set}          2.602617
{ICR - Identifying Age-Related Conditions}               2.498513
{IEEE-CIS Fraud Detection}                               2.483641
{Cassava Leaf Disease Classification}                    2.439024
{Tabular P

In [ ]:
# Undersample the male data to match the number of female samples
undersampled_male_data = resample(male_data, replace=False,
                                  n_samples= NUMBER_OFF_SAMPLES_BOTH_GENDERS - len(female_data),
                                  stratify= male_data.top_labels , random_state=42)

# Concatenate the undersampled male data with the original female data
filtered_df = pd.concat([female_data, undersampled_male_data])

# Shuffle the dataframe to mix the samples
filtered_df = filtered_df.sample(frac=1, random_state=42)

In [ ]:
undersampled_male_data.top_labels.value_counts() / len(undersampled_male_data) * 100

{Titanic - Machine Learning from Disaster}              29.225023
{House Prices - Advanced Regression Techniques}          8.683473
{Digit Recognizer}                                       8.309991
{Natural Language Processing with Disaster Tweets}       5.228758
{CommonLit Readability Prize}                            4.014939
{Housing Prices Competition for Kaggle Learn Users}      3.828198
{Spaceship Titanic}                                      3.548086
{Mechanisms of Action (MoA) Prediction}                  3.361345
{ICR - Identifying Age-Related Conditions}               3.081232
{Iris Species}                                           2.801120
{IEEE-CIS Fraud Detection}                               2.801120
{Cassava Leaf Disease Classification}                    2.707750
{Tabular Playground Series - Nov 2021}                   2.614379
{SIIM-ISIC Melanoma Classification}                      2.614379
{Jane Street Market Prediction}                          2.334267
{Credit Ca

In [ ]:
filtered_df.groupby(['Gender']).size() / NUMBER_OFF_SAMPLES_BOTH_GENDERS * 100

Gender
female    46.45
male      53.55
dtype: float64

In [ ]:
filtered_df.reset_index(drop=True, inplace = True)

In [ ]:
filtered_df.to_csv(f'gender_undersampled_codes.csv')